### Reevaluate Results

The results that were created with test_base.test_similarities should be reevaluated with other hyperparameters.

Mainly with varying cosine sim

In [1]:
from typing import List
import pandas as pd
import os
import sys
import tqdm
import numpy as np
import ast
import glob
import test_base

In [2]:
df_nace_codes_descriptions = pd.read_csv("../data/NACE_Rev2_Structure_Explanatory_Notes_EN__1_.tsv", sep="\t")

In [3]:
results_path = "/Users/hendrikweichel/projects/NaceCodeClassification/nace_report_topic_analysis_3/results/dataset__reports_subset_from_full_data_1_sentence_len_6__min_chunk_len_100__cos_thresh_0.4__nace_level_4"

In [4]:
reports_results = glob.glob(results_path+ "/*/*_long.csv")

In [5]:
dataset_path = "../data/datasets/german_annual_reports"
dataset_path = "../data/datasets/stoxx_600"
dataset_path = "../data/datasets/stoxx_600_extended"
dataset_path = "../data/datasets/reports_subset_from_full_data_1"

In [6]:
over_view_df_path = os.path.join(dataset_path, os.path.basename(dataset_path) + "_overview.csv")

dataset_path_texts = os.path.join(dataset_path, "TXTs")

dataset_name = os.path.basename(dataset_path)

In [7]:
nace_classes = pd.read_csv(over_view_df_path, index_col=0)
nace_classes.head()

,Symbol,Name,Company is Active,Company Founded Date,Country of Primary Listing Iso3,CUSIP,Date Of First Trade,Entity Country HQ,Entity Credit Parent,NACE,...,ISIN,Primary Equity Listing,Proper Name,Public Company,Region Ticker,Sec is Primary Issue,Sec Type,SEDOL,NACE_letter,Report
5789,ZW0009011041,Ariston Holdings Ltd.,1,1947.0,ZWE,V97772103,20090317.0,ZWE,@NA,1.19,...,ZW0009011041,603408,Ariston Holdings Ltd.,1.0,ARIS-ZW,1,SHARE,6034081,A,Ariston Holdings Ltd.1.pdf
35816,INE978A01027,Heritage Foods Limited,1,1992.0,IND,Y3179H146,20020117.0,IND,06FQLY-E,1.41,...,INE978A01027,BF2F40,Heritage Foods Limited,1.0,519552-IN,1,SHARE,BF2F405,A,Heritage Foods Limited1.pdf
80373,MYL7854OO002,Timberwell Bhd.,1,1996.0,MYS,Y88399103,19970516.0,MYS,05JH15-E,2.30,...,MYL7854OO002,690556,Timberwell Bhd.,1.0,7854-MY,1,SHARE,6905563,A,Timberwell Bhd.1.pdf
49813,MYQ0189OO009,Matang Bhd.,1,2015.0,MYS,Y58347108,20170117.0,MYS,@NA,1.19,...,MYQ0189OO009,BYYQB5,Matang Bhd.,1.0,0189-MY,1,SHARE,BYYQB53,A,Matang Bhd.2.pdf
73064,MYL4316OO005,Sin Heng Chan (Malaya) Bhd.,1,1962.0,MYS,Y80178109,19880324.0,MYS,05YMQ5-E,1.19,...,MYL4316OO005,681088,Sin Heng Chan (Malaya) Bhd.,1.0,4316-MY,1,SHARE,6810883,A,Sin Heng Chan (Malaya) Bhd.1.pdf


In [8]:
report_to_nace_class = nace_classes.dropna(subset=["Report"]).set_index('Report').to_dict()["NACE"]
report_to_nace_class

{'Ariston Holdings Ltd.1.pdf': 1.19,
 'Heritage Foods Limited1.pdf': 1.41,
 'Timberwell Bhd.1.pdf': 2.3,
 'Matang Bhd.2.pdf': 1.19,
 'Sin Heng Chan (Malaya) Bhd.1.pdf': 1.19,
 'Tech-bank Food Co., Ltd.3.pdf': 1.46,
 'Namoi Cotton Ltd1.pdf': 1.63,
 'Atlantic Sapphire ASA1.pdf': 3.11,
 'Agra Limited2.pdf': 1.19,
 'Huisheng International Holdings Ltd.3.pdf': 1.46,
 'Australian Agricultural Company Limited1.pdf': 1.62,
 'Waterbase Limited2.pdf': 3.21,
 'CannAmerica Brands Corp.1.pdf': 1.3,
 'Qian Hu Corporation Limited1.pdf': 3.21,
 'Jawala Inc.1.pdf': 1.19,
 'Green Thumb Industries Inc.1.pdf': 1.19,
 'CLS Holdings USA Inc2.pdf': 1.19,
 'China Bozza Development Holdings Limited1.pdf': 2.4,
 'Salmon Evolution ASA1.pdf': 3.21,
 'North American Cannabis Holdings, Inc.1.pdf': 1.19,
 'Malwatte Valley Plantations Plc1.pdf': 1.61,
 'Greenheart Group Limited1.pdf': 2.2,
 'Bumitama Agri Ltd.1.pdf': 1.19,
 'Genus plc1.pdf': 1.62,
 'Kotagala Plantations Plc1.pdf': 2.3,
 'PT Andira Agro Tbk1.pdf': 1.1

In [9]:
def evaluate_for_new_cos_thresh(cos_threshold = 0.4):
    recording = [] 

    for reports_result in tqdm.tqdm(reports_results): 

#        print("Report: ", reports_result)

        report_name = os.path.basename(reports_result.split("/")[-2]).replace(".txt",".pdf")

        df_similarities = pd.read_csv(reports_result)

        scores_column_names = [column for column in df_similarities.columns if "Scores" in column]

        # apply threshold on similarities
        df_temp = df_similarities[scores_column_names][df_similarities[scores_column_names] > cos_threshold]    

        # replace na vals with 0
        df_temp = df_temp.fillna(0)

        # get mean values for each nace code
        mean_vals = df_temp.mean().sort_values(ascending=False)

        # record the results
        mean_vals_dict = {k[7:]:round(v,3) for k,v in mean_vals.to_dict().items()}
        
        # get label of the report
        label = report_to_nace_class.get(report_name)

        try: 
            if (isinstance(label, float) or isinstance(label, int)) and len(str(label).split(".")[0]) == 1: 
            #         temp_label = "0" + str(label) 
            # else: 
            #         temp_label = str(label) 
                label = "0" + str(label) 
            label_description = df_nace_codes_descriptions[df_nace_codes_descriptions["CODE"] == str(label)]["NAME"].iloc[0]
            evaluation = test_base.get_evaluation(mean_vals_dict, label, df_nace_codes_descriptions)
        except IndexError: 
            label_description = ""
            evaluation = {}

        recording.append({"name": os.path.basename(report_name), "NACE": label, "label_description": label_description, **evaluation})

        df_recording = pd.DataFrame(recording)

    return df_recording

In [10]:
#for cos_thres in [0,0.1,0.2,0.3,0.35,0.4,0.45,0.5,0.55,0.6,0.65,0.7,0.75]:
for cos_thres in [0.4]:
    df_recording = evaluate_for_new_cos_thresh(cos_thres)
    print("Cos Thresh: ", cos_thres, " Mean res: ", df_recording["position_lvl_1"].mean())

100%|██████████| 339/339 [03:47<00:00,  1.49it/s]

Cos Thresh:  0.4  Mean res:  3.584070796460177


In [11]:
df_recording.to_csv(results_path + "/recordings.csv")